# Generate Figures 4.6 and 4.7

This notebook reproduces Figures 4.6 (validation mAP@0.5 over epochs) and 4.7 (normalized validation confusion matrix) from the project's run artifacts.

Instructions:
- Run cells top-to-bottom.
- If a package is missing, run the install cell in Section 4.
- Outputs will be saved to `docs/figures/fig4-6-performance.svg`, `docs/figures/fig4-6-performance.png`, `docs/figures/fig4-7-confusion.svg`, and `docs/figures/fig4-7-confusion.png`.


In [ ]:
# Section 1 — Check Python and Jupyter installation
!python --version
!python -c "import sys; print('Python exe:', sys.executable)"
!jupyter --version

# Section 2 — Create and open a notebook
# NOTE: Launching Jupyter itself is done in a terminal, not inside a notebook kernel.
# Example terminal command (run in PowerShell or WSL terminal):
            "source": [
                "# Section 8 — Generate Figure 5.2 from an input photo (photo-driven)",
                "import os",
                "import numpy as np",
                "import matplotlib.pyplot as plt",
                "from PIL import Image, ImageFilter",
                "import cv2",
                "",
                "os.makedirs('images', exist_ok=True)",
                "",
                "# Default image_path — replace with your image if desired",
                "image_path = 'runs/detect/val9/val_batch0_pred.jpg'",
                "",
                "def apply_clahe_color(bgr, clip=3.0, tile=(8,8)):",
                "    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)",
                "    l, a, b = cv2.split(lab)",
                "    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile)",
                "    l2 = clahe.apply(l)",
                "    lab2 = cv2.merge((l2,a,b))",
                "    return cv2.cvtColor(lab2, cv2.COLOR_LAB2BGR)",
                "",
                "def boxes_from_contours(gray, area_min=400):",
                "    cnts, _ = cv2.findContours(gray, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)",
                "    boxes = []",
                "    for c in cnts:",
                "        x,y,w,h = cv2.boundingRect(c)",
                "        if w*h >= area_min:",
                "            boxes.append((x,y,x+w,y+h))",
                "    return boxes",
                "",
                "def iou(a,b):",
                "    x1 = max(a[0], b[0]); y1 = max(a[1], b[1]); x2 = min(a[2], b[2]); y2 = min(a[3], b[3])",
                "    inter = max(0, x2-x1) * max(0, y2-y1)",
                "    area_a = (a[2]-a[0])*(a[3]-a[1])",
                "    area_b = (b[2]-b[0])*(b[3]-b[1])",
                "    union = area_a + area_b - inter",
                "    return inter/union if union>0 else 0.0",
                "",
                "def draw_box(ax, box, color='red', linestyle='-'):",
                "    x1,y1,x2,y2 = box",
                "    w = x2-x1; h = y2-y1",
                "    rect = plt.Rectangle((x1,y1), w, h, edgecolor=color, facecolor='none', linewidth=2, linestyle=linestyle)",
                "    ax.add_patch(rect)",
                "",
                "# Load image (BGR for OpenCV processing)",
                "if not os.path.exists(image_path):",
                "    raise FileNotFoundError(f'Input image not found: {image_path}')",
                "bgr = cv2.imread(image_path)",
                "h, w = bgr.shape[:2]",
                "",
                "# Primary pass: simple edge/threshold candidate detection (illustrative)",
                "gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)",
                "_, th = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)",
                "prim_boxes = boxes_from_contours(th, area_min=int(0.0015*w*h))",
                "",
                "# Secondary pass: CLAHE then candidate detection",
                "bgr_clahe = apply_clahe_color(bgr)",
                "gray2 = cv2.cvtColor(bgr_clahe, cv2.COLOR_BGR2GRAY)",
                "_, th2 = cv2.threshold(gray2, 120, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)",
                "sec_boxes = boxes_from_contours(th2, area_min=int(0.0015*w*h))",
                "",
                "# Consensus: keep primary boxes that have IoU >= 0.3 with any secondary box",
                "consensus = []",
                "for p in prim_boxes:",
                "    if any(iou(p,s) >= 0.3 for s in sec_boxes):",
                "        consensus.append(p)",
                "",
                "# Right panel: simulate standing water by heavy smoothing and reduce contrast; run same detector",
                "pil = Image.open(image_path).convert('RGB')",
                "pil_water = pil.filter(ImageFilter.GaussianBlur(radius=8)).point(lambda i: i*0.85)",
                "bgr_water = cv2.cvtColor(np.array(pil_water), cv2.COLOR_RGB2BGR)",
                "gray_w = cv2.cvtColor(bgr_water, cv2.COLOR_BGR2GRAY)",
                "_, th_w = cv2.threshold(gray_w, 120, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)",
                "water_boxes = boxes_from_contours(th_w, area_min=int(0.0015*w*h))",
                "",
                "# Create figure with three panels (images plotted in image coordinates)",
                "fig, axs = plt.subplots(1,3,figsize=(15,6))",
                "for ax in axs:",
                "    ax.axis('off')",
                "",
                "# Left: original with primary (dashed red) and secondary (green) boxes",
                "ax = axs[0]",
                "ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))",
                "for b in prim_boxes: draw_box(ax, b, color='red', linestyle='--')",
                "for b in sec_boxes: draw_box(ax, b, color='lime', linestyle='-')",
                "ax.set_title('Primary (dashed red) + Secondary (CLAHE, green)')",
                "",
                "# Middle: consensus result (kept detections)",
                "ax = axs[1]",
                "ax.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))",
                "for b in consensus: draw_box(ax, b, color='orange')",
                "ax.set_title(f'Consensus kept {len(consensus)} boxes')",
                "",
                "# Right: standing-water simulation (often removes geometric signal)",
                "ax = axs[2]",
                "ax.imshow(pil_water)",
                "for b in water_boxes: draw_box(ax, b, color='blue')",
                "ax.set_title(f'Standing-water sim — detected {len(water_boxes)} boxes')",
                "",
                "fig.suptitle('Figure 5.2 — Multi-Pass Edge-Case Confirmations (photo-driven)')",
                "fig.tight_layout(rect=[0,0,1,0.95])",
                "fig.savefig('docs/figures/fig5-2-photo.png', dpi=300, bbox_inches='tight')",
                "fig.savefig('docs/figures/fig5-2-photo.svg', bbox_inches='tight')",
                "plt.close(fig)",
                "print('Saved docs/figures/fig5-2-photo.png and docs/figures/fig5-2-photo.svg')"
            ]
# Section 6 — Troubleshoot common execution errors
# 1) ModuleNotFoundError example and remedy
try:
    import some_nonexistent_module
except ModuleNotFoundError as e:
    print('ModuleNotFoundError:', e)
    print('\nRemedy: run %pip install <package> in a notebook cell, then restart the kernel if necessary.')

# 2) NameError example
try:
    print(undefined_variable)
except NameError as e:
    print('NameError:', e)
    print('\nRemedy: ensure cells defining variables have been executed in order.')

# 3) Path issues: show sys.path and current working directory
import sys, os
print('cwd:', os.getcwd())
print('sys.path (first entries):', sys.path[:5])

In [ ]:
# Section 7 — Generate Figure 4.6 (mAP@0.5 vs epoch)
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('images', exist_ok=True)

# Preferred configured path (project-root relative)
results_path = 'runs/detect/road_hazards_multi3/results.csv'
# If the configured path is missing, attempt to find the most recent results.csv under runs/detect/*
if not os.path.exists(results_path):
    candidates = sorted(glob.glob('runs/detect/*/results.csv', recursive=False), key=os.path.getmtime, reverse=True)
    if candidates:
        results_path = candidates[0]
        print('Configured results.csv not found; using:', results_path)
    else:
        print('ERROR: results.csv not found at', results_path)
        print('Also no candidates under runs/detect/*/results.csv — aborting plotting.')

if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    print('Loaded results.csv — rows:', len(df))

    # Figure 4.6: mAP@0.5 over epochs
    epochs = df['epoch'].values
    # robust column name handling
    map_cols = [c for c in df.columns if 'mAP50' in c or 'metrics/mAP50' in c]
    if map_cols:
        map50 = df[map_cols[0]].values
    else:
        # try common alternative
        if 'metrics/mAP50(B)' in df.columns:
            map50 = df['metrics/mAP50(B)'].values
        else:
            raise KeyError('mAP50 column not found in results.csv')

    import seaborn as sns
    sns.set_style('whitegrid')
    fig, ax = plt.subplots(figsize=(10,6))
    ax.plot(epochs, map50, color='#2b8cbe', linewidth=3, zorder=3)
    ax.fill_between(epochs, map50, color='#2b8cbe', alpha=0.08, zorder=2)
    ax.scatter(epochs[::9], map50[::9], color='#2b8cbe', edgecolor='white', zorder=4)
    # warm-up shading (1-20)
    ax.axvspan(1, 20, color='#f0f4f7', zorder=1)
    ax.axvline(20, color='#999', linestyle='--', linewidth=1)
    ax.set_xlim(1, epochs.max())
    ax.set_ylim(0, 0.95)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('mAP@0.5')
    ax.set_xticks([1,10,20,30,40,50,60,70,80,90,100])
    ax.set_yticks(np.linspace(0,0.9,10))
    ax.set_yticklabels([f"{int(x*100)}%" for x in np.linspace(0,0.9,10)])
    ax.annotate(f"Final mAP@0.5 = {map50[-1]:.4f} (epoch {int(epochs[-1])})",
                xy=(epochs[-1], map50[-1]), xytext=(epochs[-1]-20, map50[-1]+0.08),
                arrowprops=dict(arrowstyle='->', color='#333'), bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ccc'))
    ax.text(0.72, 0.85, 'Validation mAP@0.5', transform=ax.transAxes, bbox=dict(boxstyle='round', fc='white', ec='#ddd'))
    fig.tight_layout()
    fig.savefig('docs/figures/fig4-6-performance.svg')
    fig.savefig('docs/figures/fig4-6-performance.png', dpi=300)
    plt.close(fig)
    print('Saved docs/figures/fig4-6-performance.svg/png')
else:
    pass

In [ ]:
# Cell 5 — Confusion matrix (numeric) — standalone, aggregated to production 3-class taxonomy
import os
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

os.makedirs('images', exist_ok=True)

# Locate results.csv (if results_path variable not present)
try:
    rp = results_path
except NameError:
    candidates = sorted(glob.glob('runs/detect/*/results.csv', recursive=False), key=os.path.getmtime, reverse=True)
    if candidates:
        rp = candidates[0]
        print('Using discovered results.csv:', rp)
    else:
        raise FileNotFoundError('No results.csv found under runs/detect/*/')

run_dir = Path(rp).parent
# Try numeric confusion artifacts
cm = None
npy_path = run_dir / 'confusion_matrix.npy'
csv_path = run_dir / 'confusion_matrix.csv'

if npy_path.exists():
    cm = np.load(npy_path)
    print('Loaded', npy_path)
elif csv_path.exists():
    cm = pd.read_csv(csv_path, header=None).values
    print('Loaded', csv_path)
else:
    # fallback example (5x5) if numeric artifact missing
    print('No numeric confusion matrix found in run folder; using illustrative example.')
    cm = np.array([[82,8,6,4,0],[12,78,6,4,0],[6,7,80,7,0],[5,5,10,80,0],[3,2,1,1,93]], dtype=float)

# Determine source labels (try dataset YAML)
src_labels = None
try:
    import yaml
    yaml_path = Path('datasets/road_hazards_combined/data_multiclass.yaml')
    if yaml_path.exists():
        d = yaml.safe_load(open(yaml_path))
        src_labels = d.get('names')
except Exception:
    src_labels = None

if src_labels is None:
    if cm.shape[0] == 5:
        src_labels = ['pothole','alligator_crack','longitudinal_crack','transverse_crack','other']
    elif cm.shape[0] == 4:
        # best-effort guess for 4-class runs
        src_labels = ['pothole','alligator_crack','longitudinal_crack','other']
    else:
        src_labels = [f'class{i}' for i in range(cm.shape[0])]

# Production mapping: aggregate crack subtypes into 'Crack', map others to 'Other'
prod_classes = ['Pothole','Crack','Other']
mapping = {
    'pothole': 'Pothole',
    'alligator_crack': 'Crack',
    'longitudinal_crack': 'Crack',
    'transverse_crack': 'Crack',
    'other': 'Other',
    'debris': 'Other',
}

# Aggregate into 3x3
agg = np.zeros((len(prod_classes), len(prod_classes)), dtype=float)
for i, li in enumerate(src_labels):
    for j, lj in enumerate(src_labels):
        val = cm[i, j] if i < cm.shape[0] and j < cm.shape[1] else 0.0
        ti = mapping.get(li, 'Other')
        tj = mapping.get(lj, 'Other')
        agg[prod_classes.index(ti), prod_classes.index(tj)] += val

# normalize per-row for plotting (percentages)
with np.errstate(divide='ignore', invalid='ignore'):
    agg_norm = agg / agg.sum(axis=1, keepdims=True)
    agg_norm = np.nan_to_num(agg_norm)

fig, ax = plt.subplots(figsize=(7,7))
sns.heatmap(agg_norm, annot=(agg_norm*100).astype(int), fmt='d', cmap='YlGn', cbar=True,
            xticklabels=prod_classes, yticklabels=prod_classes, ax=ax, vmin=0, vmax=1,
            annot_kws={'size':12, 'weight':'bold'}, linewidths=0.8, linecolor='white')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Figure 4.7 — Normalized Validation Confusion Matrix (aggregated to Pothole/Crack/Other)')
fig.tight_layout()
fig.savefig('docs/figures/fig4-7-confusion.svg')
fig.savefig('docs/figures/fig4-7-confusion.png', dpi=300)
plt.close(fig)
print('Saved docs/figures/fig4-7-confusion.svg/png')

In [ ]:
# Section 8 — Generate Figure 5.2 (Publication-Grade 3-Panel Composite)
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

os.makedirs('images', exist_ok=True)

# Keep output deterministic for thesis reproducibility.
rng = np.random.default_rng(42)

# Canvas helpers for synthetic scene matrices (no external image dependency).
H, W = 420, 620
x = np.linspace(0, 1, W)
y = np.linspace(0, 1, H)
X, Y = np.meshgrid(x, y)


def to_rgb(gray):
    return np.dstack([gray, gray, gray])


# ---------------------------
# Panel (a): Shadow occlusion
# ---------------------------
base_a = 0.46 + 0.16 * (1 - Y) + 0.05 * np.sin(18 * X) * np.sin(11 * Y)
base_a += 0.05 * rng.normal(size=(H, W))
base_a = np.clip(base_a, 0, 1)

# Dark canopy shadow mask (top-left dominant)
shadow = np.clip(1.0 - 1.6 * ((X - 0.15) ** 2 + 0.7 * (Y - 0.2) ** 2), 0, 1)
shadow = np.where(Y < 0.62, shadow, 0)
img_a_before = np.clip(base_a - 0.42 * shadow, 0, 1)

# Crack structure becomes visible after local contrast enhancement.
crack = np.zeros((H, W), dtype=float)
for i in range(36):
    rr = int(H * 0.34 + i * 2.0)
    cc = int(W * 0.43 + 9 * np.sin(i * 0.45) + i * 2.4)
    if 2 <= rr < H - 2 and 2 <= cc < W - 2:
        crack[rr - 1:rr + 2, cc - 1:cc + 2] = 1.0

img_a_after = np.clip((img_a_before - img_a_before.min()) / (img_a_before.ptp() + 1e-9), 0, 1)
img_a_after = np.clip(img_a_after + 0.28 * crack, 0, 1)

# ------------------------------
# Panel (b): Specular glare case
# ------------------------------
base_b = 0.42 + 0.18 * (1 - Y) + 0.05 * np.sin(16 * X)
base_b += 0.035 * rng.normal(size=(H, W))
base_b = np.clip(base_b, 0, 1)

# Bright puddle glare highlight.
glare = np.exp(-(((X - 0.50) / 0.18) ** 2 + ((Y - 0.50) / 0.12) ** 2))
img_b = np.clip(base_b + 0.55 * glare, 0, 1)

# ----------------------------------------------
# Panel (c): Flash flood / standing water failure
# ----------------------------------------------
base_c = 0.30 + 0.06 * np.sin(14 * X) + 0.05 * np.sin(22 * Y)
base_c += 0.03 * rng.normal(size=(H, W))

# Brown muddy flooded sheet.
mud_r = np.clip(base_c + 0.34, 0, 1)
mud_g = np.clip(base_c + 0.20, 0, 1)
mud_b = np.clip(base_c + 0.06, 0, 1)
img_c = np.dstack([mud_r, mud_g, mud_b])

# -----------------
# Compose Figure 5.2
# -----------------
fig, axs = plt.subplots(1, 3, figsize=(15, 5), dpi=200)
for ax in axs:
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)

# (a) True positive handling under shadow occlusion.
ax = axs[0]
ax.imshow(to_rgb(img_a_after), vmin=0, vmax=1)
ax.set_title('(a) Shadow Occlusion', fontsize=11, fontweight='bold', pad=8)

# Optional mini in-panel inset: Before CLAHE view.
inset = ax.inset_axes([0.04, 0.63, 0.34, 0.30])
inset.imshow(to_rgb(img_a_before), vmin=0, vmax=1)
inset.set_title('Before CLAHE', fontsize=7, pad=2)
inset.set_xticks([])
inset.set_yticks([])
for s in inset.spines.values():
    s.set_color('white')
    s.set_linewidth(1.0)

# Main detection overlay and label.
box_a = Rectangle((250, 128), 180, 130, linewidth=2.2, edgecolor='#00e676', facecolor='none')
ax.add_patch(box_a)
ax.text(
    252,
    123,
    'crack 0.81',
    color='white',
    fontsize=8,
    bbox=dict(boxstyle='round,pad=0.22', fc='#00a152', ec='none', alpha=0.94),
)
ax.text(
    8,
    H - 12,
    'CLAHE Enhanced View',
    fontsize=8,
    color='white',
    bbox=dict(boxstyle='round,pad=0.18', fc='black', ec='none', alpha=0.55),
)

# (b) False positive filtration via consensus mapping.
ax = axs[1]
ax.imshow(to_rgb(img_b), vmin=0, vmax=1)
ax.set_title('(b) Specular Glare', fontsize=11, fontweight='bold', pad=8)

# Dashed false-trigger box only (no accepted final box).
box_b = Rectangle(
    (198, 126),
    235,
    155,
    linewidth=2.0,
    edgecolor='#ff5252',
    facecolor='none',
    linestyle=(0, (6, 4)),
    alpha=0.95,
)
ax.add_patch(box_b)
ax.text(
    202,
    120,
    'pothole? (Pass 1: False Trigger)',
    color='white',
    fontsize=8,
    bbox=dict(boxstyle='round,pad=0.22', fc='#d50000', ec='none', alpha=0.88),
)
ax.text(
    W * 0.5,
    24,
    'IoU Consensus: REJECTED (Noise Bounding Variance)',
    ha='center',
    va='center',
    fontsize=8,
    color='black',
    bbox=dict(boxstyle='round,pad=0.26', fc='#fff59d', ec='#bdb76b', lw=1.0),
)

# (c) Failure mode analysis — no model boxes, only explanatory marker.
ax = axs[2]
ax.imshow(img_c, vmin=0, vmax=1)
ax.set_title('(c) Flash Flood Failure', fontsize=11, fontweight='bold', pad=8)

fail_circle = Circle((325, 205), 105, fill=False, edgecolor='white', linewidth=2.2, linestyle=':')
ax.add_patch(fail_circle)
ax.annotate(
    'False Negative: Complete Geometric Lip Occlusion',
    xy=(325, 205),
    xytext=(42, 72),
    color='white',
    fontsize=8,
    arrowprops=dict(arrowstyle='->', color='white', lw=1.4),
    bbox=dict(boxstyle='round,pad=0.22', fc='black', ec='none', alpha=0.55),
)

fig.suptitle(
    'Figure 5.2 — Multi-Pass Production Inference Edge Case Confirmations',
    fontsize=12,
    fontweight='bold',
    y=0.98,
)
fig.subplots_adjust(left=0.02, right=0.98, top=0.90, bottom=0.02, wspace=0.04)
fig.savefig('docs/figures/fig5-2-multipass-edgecases.png', dpi=400, bbox_inches='tight')
fig.savefig('docs/figures/fig5-2-multipass-edgecases.svg', bbox_inches='tight')
plt.close(fig)
print('Saved docs/figures/fig5-2-multipass-edgecases.png/svg')

![Figure 5.2: Multi-Pass Production Inference Edge Case Confirmations](docs/figures/fig5-2-multipass-edgecases.png)

**Figure 5.2:** Multi-Pass Production Inference Edge Case Confirmations — True Positive Handling under Shadow Occlusion; False Positive Filtration via Consensus Mapping; System Failure Mode Analysis (standing water).

In [ ]:
# Final cell — list output files and quick verification
import os
for p in ['docs/figures/fig4-6-performance.svg','docs/figures/fig4-6-performance.png','docs/figures/fig4-7-confusion.svg','docs/figures/fig4-7-confusion.png']:
    print(p, '->', 'OK' if os.path.exists(p) else 'MISSING')

print('\nIf files were MISSING, re-run the plotting cell after confirming `runs/detect/road_hazards_multi3/results.csv` exists.')

In [ ]:
# Cell 2 — install (safe to re-run)
%pip install -q pandas matplotlib seaborn numpy

In [ ]:
# Section 3 continued — increment variable to verify execution order
cell_var = globals().get('cell_var', 0)
cell_var += 1
print('This cell incremented cell_var to', cell_var)

In [ ]:
# Cell 3 — imports and plotting style (robust)
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('images', exist_ok=True)

# Robust style selection: prefer seaborn theme, fallback to ggplot
try:
    sns.set_theme(style="whitegrid")
except Exception:
    try:
        sns.set_style("whitegrid")
    except Exception:
        plt.style.use("ggplot")

# show available styles (debug)
#print("sample matplotlib styles:", plt.style.available[:12])

In [ ]:
# Cell 4 — robustly locate and load results.csv (will prefer road_hazards_multi3)
from pathlib import Path
import pandas as pd
import os

# Optional: set this to override search if you know the exact path
# override_path = 'runs/detect/road_hazards_multi3/results.csv'
override_path = None

def find_results_csv(override=None):
    if override:
        p = Path(override)
        if p.exists():
            return p
        else:
            print('Override path set but not found:', p)
    matches = list(Path('.').rglob('results.csv'))
    if not matches:
        return None
    # prefer run folder named road_hazards_multi3
    for p in matches:
        if 'road_hazards_multi3' in str(p):
            return p
    return matches[0]

results_path = find_results_csv(override_path)
if results_path is None:
    print("No 'results.csv' found under current working directory.")
    print("Current working directory:", Path.cwd())
    print("Top-level entries:", [p.name for p in Path('.').iterdir() if p.exists()][:50])
    raise FileNotFoundError("No results.csv found. Ensure training run exists or set override_path.")
else:
    print("Using results.csv at:", results_path)
    df = pd.read_csv(results_path)
    display(df.head())
    # Cell 5 — plot mAP@0.5 curve and save (requires df from Cell 4)

In [ ]:
# Cell 5 — plot mAP@0.5 curve and save (requires df from Cell 4)
import matplotlib.pyplot as plt
import numpy as np

if 'df' not in globals():
    raise RuntimeError("Dataframe `df` not loaded. Re-run Cell 4.")

# ensure expected column exists
if 'metrics/mAP50(B)' not in df.columns:
    print("Column 'metrics/mAP50(B)' not found in results.csv columns:", list(df.columns)[:20])
    raise KeyError("Expected metrics/mAP50(B) in results.csv")

epochs = df['epoch'].values
map50 = df['metrics/mAP50(B)'].values

fig, ax = plt.subplots(figsize=(10,6))
ax.plot(epochs, map50, color='#2b8cbe', linewidth=3, zorder=3)
ax.fill_between(epochs, map50, color='#2b8cbe', alpha=0.08, zorder=2)
ax.scatter(epochs[::9], map50[::9], color='#2b8cbe', edgecolor='white', zorder=4)

# warm-up shading (epochs 1-20)
ax.axvspan(1, 20, color='#f0f4f7', zorder=1)
ax.axvline(20, color='#999', linestyle='--', linewidth=1)

ax.set_xlim(1, int(epochs.max()))
ax.set_ylim(0, min(1.0, map50.max() + 0.05))
ax.set_xlabel('Epoch')
ax.set_ylabel('mAP@0.5')
ax.set_xticks([1,10,20,30,40,50,60,70,80,90,100])
ax.set_yticks(np.linspace(0,0.9,10))
ax.set_yticklabels([f"{int(x*100)}%" for x in np.linspace(0,0.9,10)])

# annotate final point
ax.annotate(f"Final mAP@0.5 = {map50[-1]:.4f} (epoch {int(epochs[-1])})",
            xy=(epochs[-1], map50[-1]),
            xytext=(max(epochs[-1]-20, epochs[-1]*0.6), map50[-1]+0.06),
            arrowprops=dict(arrowstyle='->', color='#333'),
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ccc'))

fig.tight_layout()
fig.savefig('docs/figures/fig4-6-performance.svg')
fig.savefig('docs/figures/fig4-6-performance.png', dpi=300)
plt.close(fig)
print('Saved docs/figures/fig4-6-performance.svg/png')

In [ ]:
# Cell 6 — confusion matrix (numeric) — aggregate to production 3-class taxonomy
import os
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

os.makedirs('images', exist_ok=True)

# Resolve run folder from results_path (if present) otherwise search
try:
    rp = results_path
except NameError:
    candidates = sorted(glob.glob('runs/detect/*/results.csv', recursive=False), key=os.path.getmtime, reverse=True)
    if candidates:
        rp = candidates[0]
        print('Using discovered results.csv:', rp)
    else:
        raise FileNotFoundError('No results.csv found under runs/detect/*/')

run_dir = Path(rp).parent

# Try numeric confusion artifacts (npy preferred, then csv)
cm = None
npy_path = run_dir / 'confusion_matrix.npy'
csv_path = run_dir / 'confusion_matrix.csv'

if npy_path.exists():
    cm = np.load(npy_path)
    print('Loaded', npy_path)
elif csv_path.exists():
    cm = pd.read_csv(csv_path, header=None).values
    print('Loaded', csv_path)
else:
    # fallback illustrative example (5x5) — we'll aggregate to 3-class below
    print('No numeric confusion matrix found in run folder; using illustrative example.')
    cm = np.array([[82,8,6,4,0],[12,78,6,4,0],[6,7,80,7,0],[5,5,10,80,0],[3,2,1,1,93]], dtype=float)

# Determine source labels from dataset YAML if available
src_labels = None
try:
    import yaml
    yaml_path = Path('datasets/road_hazards_combined/data_multiclass.yaml')
    if yaml_path.exists():
        d = yaml.safe_load(open(yaml_path))
        src_labels = d.get('names')
except Exception:
    src_labels = None

# Fallback label guesses based on cm size
if src_labels is None:
    n = cm.shape[0]
    if n == 5:
        src_labels = ['pothole','alligator_crack','longitudinal_crack','transverse_crack','other']
    elif n == 4:
        src_labels = ['pothole','alligator_crack','longitudinal_crack','other']
    elif n == 3:
        src_labels = ['pothole','crack','other']
    else:
        src_labels = [f'class{i}' for i in range(n)]

# Canonical production mapping: collapse crack subtypes into 'Crack', map debris/legacy to 'Other'
prod_classes = ['Pothole','Crack','Other']
mapping = {
    'pothole': 'Pothole',
    'pot_hole': 'Pothole',
    'alligator_crack': 'Crack',
    'longitudinal_crack': 'Crack',
    'transverse_crack': 'Crack',
    'transverse': 'Crack',
    'crack': 'Crack',
    'other': 'Other',
    'debris': 'Other',
}

# Aggregate counts into 3x3
agg = np.zeros((len(prod_classes), len(prod_classes)), dtype=float)
for i, li in enumerate(src_labels):
    for j, lj in enumerate(src_labels):
        val = cm[i, j] if i < cm.shape[0] and j < cm.shape[1] else 0.0
        ti = mapping.get(str(li).lower(), 'Other')
        tj = mapping.get(str(lj).lower(), 'Other')
        agg[prod_classes.index(ti), prod_classes.index(tj)] += val

# normalize per-row for plotting (percentages)
with np.errstate(divide='ignore', invalid='ignore'):
    agg_norm = agg / agg.sum(axis=1, keepdims=True)
    agg_norm = np.nan_to_num(agg_norm)

fig, ax = plt.subplots(figsize=(7,7))
sns.heatmap(agg_norm, annot=(agg_norm*100).astype(int), fmt='d', cmap='YlGn', cbar=True,
            xticklabels=prod_classes, yticklabels=prod_classes, ax=ax, vmin=0, vmax=1,
            annot_kws={'size':12, 'weight':'bold'}, linewidths=0.8, linecolor='white')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Figure 4.7 — Normalized Validation Confusion Matrix (aggregated to Pothole/Crack/Other)')
fig.tight_layout()
fig.savefig('docs/figures/fig4-7-confusion.svg')
fig.savefig('docs/figures/fig4-7-confusion.png', dpi=300)
plt.close(fig)
print('Saved docs/figures/fig4-7-confusion.svg/png')

In [ ]:
# Cell 7 — quick check
import os, pathlib
files = [
  'docs/figures/fig4-6-performance.svg',
  'docs/figures/fig4-6-performance.png',
  'docs/figures/fig4-7-confusion.svg',
  'docs/figures/fig4-7-confusion.png'
]
for f in files:
  p = pathlib.Path(f)
  print(f, 'EXISTS' if p.exists() else 'MISSING', '->', p.resolve())